# UK CPI Data Exploration
This notebook explores UK CPI inflation data from ONS.

Importing the data

In [49]:
import pandas as pd
import matplotlib.pyplot as plt
# import the csv inflation data
df = pd.read_csv("../data/raw/raw_uk_cpi.csv")
df.columns

Index(['Title', 'CPIH ANNUAL RATE 00: ALL ITEMS 2015=100'], dtype='str')

Renaming the columns

In [50]:
# rename the columns
df = df.rename(columns={'Title': 'Date', 'CPIH ANNUAL RATE 00: ALL ITEMS 2015=100': 'Inflation'})
df.columns
df.head()

,Date,Inflation
0,CDID,L55O
1,Source dataset ID,MM23
2,PreUnit,NaN
3,Unit,%
4,Release date,22-07-2026


Creating a mask for monthly data

In [51]:
# ALTERNATIVE
#months = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]

# 2. Join the list into a regex string: 'JAN|FEB|MAR...'
#regex_pattern = "|".join(months)

# 3. Pass the pattern into your mask
#month_mask = df["Date"].str.contains(regex_pattern, case=False)


# include only monthly inflation data
# .contains returns a series of boolean data
month_mask = df["Date"].str.contains("JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC")
month_mask.head()

0    False
1    False
2    False
3    False
4    False
Name: Date, dtype: bool

Creating a new monthly dataframe

In [52]:
# use boolean indexing to filter the data based on the true/false mask
month_df = df[month_mask]
month_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 194 to 643
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Date       450 non-null    str  
 1   Inflation  450 non-null    str  
dtypes: str(2)
memory usage: 7.2 KB


Create a copy of the monthly data to prepare for cleaning the data types

In [53]:
clean_df = month_df.copy()
# convert to float
clean_df["Inflation"] = clean_df["Inflation"].astype(float)

# convert to datetime
clean_df["Date"] = pd.to_datetime(clean_df["Date"], format="%Y %b")
clean_df.info()
clean_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 194 to 643
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       450 non-null    datetime64[us]
 1   Inflation  450 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 7.2 KB


,Date,Inflation
194,1989-01-01,5.7
195,1989-02-01,5.8
196,1989-03-01,5.9
197,1989-04-01,5.6
198,1989-05-01,5.9


Set a new index to be the date instead of the row number.
The clean data now: monthly observations only, dates converted, inflation values numeric, date index created & chronological structure

In [54]:
clean_df = clean_df.set_index("Date")
clean_df.info()
clean_df.head()
clean_df.index.is_monotonic_increasing

<class 'pandas.DataFrame'>
DatetimeIndex: 450 entries, 1989-01-01 to 2026-06-01
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Inflation  450 non-null    float64
dtypes: float64(1)
memory usage: 7.0 KB


True